In [1]:
!pip -q install streamlit pyngrok bcrypt PyJWT

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 50.0 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import os, re, time, jwt, bcrypt, sqlite3, smtplib, secrets, datetime
import streamlit as st
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

DB_NAME = "freightiq_auth.db"
JWT_SECRET = os.getenv("JWT_SECRET", "demo-secret")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")
ADMIN_USERNAME = "admin@portal.com"
ADMIN_PASSWORD = "AdminPassword@2026"
OTP_EXPIRY_SECONDS = 300

SECURITY_QUESTIONS = [
    "What is your favourite city?",
    "What is your pet name?",
    "What was your first school?",
    "What is your favourite shipment lane?"
]

st.set_page_config(page_title="FreightIQ Auth Console", page_icon="FQ", layout="wide", initial_sidebar_state="collapsed")

def get_connection():
    return sqlite3.connect(DB_NAME, check_same_thread=False)

def hash_text(value):
    return bcrypt.hashpw(value.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")

def check_hash(value, hashed):
    try:
        return bcrypt.checkpw(value.encode("utf-8"), hashed.encode("utf-8"))
    except Exception:
        return False

def init_db():
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("""
            CREATE TABLE IF NOT EXISTS users (
                email TEXT PRIMARY KEY,
                username TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                security_question TEXT NOT NULL,
                security_answer_hash TEXT NOT NULL,
                created_at TEXT NOT NULL,
                recent_login TEXT
            )
        """)
        c.execute("""
            CREATE TABLE IF NOT EXISTS password_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL,
                password_hash TEXT NOT NULL,
                set_at TEXT NOT NULL
            )
        """)
        conn.commit()

    demo_users = [
        ("springboardmentor018@gmail.com", "Mentor_018", "Welcome@123", "What is your favourite city?", "bengaluru"),
        ("springboardmentor038@gmail.com", "Mentor_038", "Welcome@123", "What is your pet name?", "shadow")
    ]
    for email, username, password, question, answer in demo_users:
        if not get_user_by_email(email):
            register_user(email, username, password, question, answer)

def valid_email(email):
    return re.match(r"^[A-Za-z]{2,}[\w.+-]*@[A-Za-z]{2,}\.[A-Za-z]{2,}$", email.strip()) is not None

def password_error(password):
    if len(password) < 8:
        return "Password must be at least 8 characters long."
    if not re.search(r"[A-Z]", password):
        return "Password must contain at least one uppercase letter."
    if not re.search(r"[a-z]", password):
        return "Password must contain at least one lowercase letter."
    if not re.search(r"\d", password):
        return "Password must contain at least one number."
    if not re.search(r"[^A-Za-z0-9]", password):
        return "Password must contain at least one special symbol."
    return ""

def password_strength(password):
    score = 0
    score += len(password) >= 8
    score += bool(re.search(r"[A-Z]", password) and re.search(r"[a-z]", password))
    score += bool(re.search(r"\d", password) and re.search(r"[^A-Za-z0-9]", password))
    return int(score)

def register_user(email, username, password, question, answer):
    now = datetime.datetime.now().strftime("%d %b %Y, %I:%M %p")
    pwd_hash = hash_text(password)
    ans_hash = hash_text(answer.lower().strip())
    with get_connection() as conn:
        c = conn.cursor()
        try:
            c.execute("""
                INSERT INTO users
                (email, username, password_hash, security_question, security_answer_hash, created_at, recent_login)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (email.lower().strip(), username.strip(), pwd_hash, question, ans_hash, now, "Never"))
            c.execute("INSERT INTO password_history (email, password_hash, set_at) VALUES (?, ?, ?)",
                      (email.lower().strip(), pwd_hash, now))
            conn.commit()
            return True
        except sqlite3.IntegrityError:
            return False

def get_user_by_email(email):
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT email, username, password_hash, security_question, security_answer_hash, created_at, recent_login FROM users WHERE email=?",
                  (email.lower().strip(),))
        return c.fetchone()

def get_user_by_username(username):
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT email, username, password_hash, security_question, security_answer_hash, created_at, recent_login FROM users WHERE LOWER(username)=?",
                  (username.lower().strip(),))
        return c.fetchone()

def get_user_by_identity(identity):
    identity = identity.strip()
    if "@" in identity:
        return get_user_by_email(identity)
    return get_user_by_username(identity)

def authenticate(identity, password):
    user = get_user_by_identity(identity)
    if user and check_hash(password, user[2]):
        recent = datetime.datetime.now().strftime("%d %b %Y, %I:%M %p")
        with get_connection() as conn:
            c = conn.cursor()
            c.execute("UPDATE users SET recent_login=? WHERE email=?", (recent, user[0]))
            conn.commit()
        return get_user_by_email(user[0])
    return None

def make_token(email, username):
    payload = {
        "email": email,
        "username": username,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_token(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None

def password_was_used(email, password):
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT password_hash FROM password_history WHERE email=?", (email.lower().strip(),))
        return any(check_hash(password, row[0]) for row in c.fetchall())

def update_password(email, new_password):
    now = datetime.datetime.now().strftime("%d %b %Y, %I:%M %p")
    pwd_hash = hash_text(new_password)
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("UPDATE users SET password_hash=? WHERE email=?", (pwd_hash, email.lower().strip()))
        c.execute("INSERT INTO password_history (email, password_hash, set_at) VALUES (?, ?, ?)",
                  (email.lower().strip(), pwd_hash, now))
        conn.commit()

def get_all_users():
    with get_connection() as conn:
        c = conn.cursor()
        c.execute("SELECT username, email, created_at, recent_login FROM users ORDER BY created_at DESC")
        return c.fetchall()

def send_otp_email(to_email, otp):
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        raise Exception("EMAIL_ADDRESS or EMAIL_PASSWORD missing in Colab Secrets.")

    msg = MIMEMultipart()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = to_email
    msg["Subject"] = "FreightIQ Password Reset OTP"

    html = f"""
    <html>
    <body style="font-family:Arial;background:#f8fafc;padding:24px;">
      <div style="max-width:520px;margin:auto;background:white;border:1px solid #e2e8f0;border-radius:14px;padding:24px;">
        <h2 style="color:#2563EB;">FreightIQ Password Reset OTP</h2>
        <p>Your one-time password is:</p>
        <div style="font-size:34px;font-weight:800;letter-spacing:6px;color:#0F172A;background:#e0f2fe;padding:16px;border-radius:10px;text-align:center;">
          {otp}
        </div>
        <p style="color:#64748B;">This OTP expires in 5 minutes. Do not share it with anyone.</p>
      </div>
    </body>
    </html>
    """
    msg.attach(MIMEText(html, "html"))

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.sendmail(EMAIL_ADDRESS, to_email, msg.as_string())

def load_css():
    st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700;800&display=swap');
    html, body, [class*="css"] { font-family: 'Poppins', sans-serif; }
    .stApp {
        background:
        radial-gradient(circle at top left, rgba(37,99,235,.18), transparent 35rem),
        radial-gradient(circle at bottom right, rgba(6,182,212,.20), transparent 35rem),
        #F8FAFC;
    }
    [data-testid="stSidebar"], [data-testid="collapsedControl"] { display:none; }
    #MainMenu, footer, header { visibility:hidden; }
    .hero-title {
        font-size: clamp(42px, 7vw, 82px);
        line-height: .98;
        font-weight: 800;
        color: #0F172A;
    }
    .hero-sub { color:#475569; font-size:18px; line-height:1.7; max-width:760px; }
    .eyebrow {
        color:#2563EB;
        font-weight:800;
        text-transform:uppercase;
        letter-spacing:.08em;
        font-size:13px;
    }
    .card {
        padding:24px;
        border-radius:18px;
        background:rgba(255,255,255,.78);
        border:1px solid rgba(148,163,184,.28);
        box-shadow:0 24px 70px rgba(15,23,42,.14);
        backdrop-filter:blur(20px);
    }
    .metric-grid, .feature-grid, .dash-grid {
        display:grid;
        grid-template-columns:repeat(3,1fr);
        gap:16px;
        margin-top:24px;
    }
    .metric, .feature, .dash-card {
        border:1px solid rgba(148,163,184,.28);
        border-radius:16px;
        padding:20px;
        background:rgba(255,255,255,.76);
        box-shadow:0 16px 40px rgba(15,23,42,.10);
    }
    .metric strong { color:#2563EB; font-size:24px; display:block; }
    .freight-art {
        min-height:340px;
        border-radius:24px;
        color:white;
        padding:28px;
        background:linear-gradient(135deg,#2563EB,#06B6D4);
        box-shadow:0 28px 80px rgba(37,99,235,.35);
    }
    .status-pill {
        display:inline-block;
        padding:6px 12px;
        border-radius:999px;
        background:rgba(34,197,94,.16);
        color:#15803D;
        font-weight:700;
    }
    .stButton > button {
        border-radius:12px;
        min-height:48px;
        font-weight:700;
        border:0;
        background:linear-gradient(135deg,#2563EB,#06B6D4);
        color:white;
    }
    .stButton > button:hover { color:white; transform:translateY(-2px); }
    @media(max-width:900px){
        .metric-grid,.feature-grid,.dash-grid{grid-template-columns:1fr;}
    }
    </style>
    """, unsafe_allow_html=True)

def landing_page():
    left, right = st.columns([1.2, .8], gap="large")
    with left:
        st.markdown("""
        <div class="eyebrow">Enterprise Freight Authentication</div>
        <div class="hero-title">Intelligent Freight Quote Generation System</div>
        <div class="hero-sub">
        Secure authentication module for freight quote users and administrators with signup, login,
        password recovery, JWT sessions, and admin user visibility.
        </div>
        <div class="metric-grid">
          <div class="metric"><strong>JWT</strong><span>Session handling</span></div>
          <div class="metric"><strong>OTP</strong><span>Email recovery</span></div>
          <div class="metric"><strong>Admin</strong><span>User dashboard</span></div>
        </div>
        """, unsafe_allow_html=True)
        c1, c2, c3 = st.columns(3)
        if c1.button("Login", use_container_width=True):
            st.session_state.page = "Login"; st.rerun()
        if c2.button("Signup", use_container_width=True):
            st.session_state.page = "Signup"; st.rerun()
        if c3.button("Admin Login", use_container_width=True):
            st.session_state.page = "Admin Login"; st.rerun()
    with right:
        st.markdown("""
        <div class="freight-art">
          <h2>Freight Route Quote</h2>
          <p>Origin: Bengaluru<br>Destination: Mumbai<br>Mode: Express<br>Estimated Quote: $2,840</p>
          <h1 style="font-size:54px;margin-top:60px;">Warehouse → Truck → Delivery</h1>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("""
    <div class="feature-grid">
      <div class="feature"><h3>Secure Login</h3><p>Users can login using username/email and password.</p></div>
      <div class="feature"><h3>Dual Recovery</h3><p>Password reset using security question or email OTP.</p></div>
      <div class="feature"><h3>Admin Dashboard</h3><p>Admin can view users without seeing passwords.</p></div>
    </div>
    """, unsafe_allow_html=True)

def login_page():
    st.subheader("User Login")
    st.caption("Any signed-up user can login using username or email.")
    with st.form("login_form"):
        identity = st.text_input("Username/Email")
        password = st.text_input("Password", type="password")
        remember = st.checkbox("Remember Me")
        submitted = st.form_submit_button("Login", use_container_width=True)

    if submitted:
        if not identity or not password:
            st.error("All fields are mandatory.")
        else:
            user = authenticate(identity, password)
            if user:
                st.session_state.user = {"email": user[0], "username": user[1]}
                st.session_state.jwt_token = make_token(user[0], user[1])
                st.success("Login successful. JWT session is active.")
                time.sleep(1)
                st.session_state.page = "User Dashboard"
                st.rerun()
            else:
                st.error("Invalid username/email or password.")

    c1, c2, c3 = st.columns(3)
    if c1.button("Create Account"):
        st.session_state.page = "Signup"; st.rerun()
    if c2.button("Forgot Password"):
        st.session_state.page = "Forgot Password"; st.rerun()
    if c3.button("Home"):
        st.session_state.page = "Landing Page"; st.rerun()

def signup_page():
    st.subheader("Signup")
    with st.form("signup_form"):
        c1, c2 = st.columns(2)
        username = c1.text_input("Username")
        email = c2.text_input("Email")
        password = c1.text_input("Password", type="password")
        confirm = c2.text_input("Confirm Password", type="password")
        question = st.selectbox("Security Question", [""] + SECURITY_QUESTIONS)
        answer = st.text_input("Security Answer")
        score = password_strength(password)
        st.progress(score / 3 if score else 0)
        st.caption(["Password strength", "Weak", "Medium", "Strong"][score])
        submitted = st.form_submit_button("Signup", use_container_width=True)

    if submitted:
        if not all([username, email, password, confirm, question, answer]):
            st.error("All fields are mandatory.")
        elif not valid_email(email):
            st.error("Email must be shaped like ab@cd.ef.")
        elif password_error(password):
            st.error(password_error(password))
        elif password != confirm:
            st.error("Password and Confirm Password must match.")
        elif not register_user(email, username, password, question, answer):
            st.error("Username or email already exists.")
        else:
            st.success("Signup successful. You can now login with this account.")
            time.sleep(1)
            st.session_state.page = "Login"
            st.rerun()

    if st.button("Back to Login"):
        st.session_state.page = "Login"; st.rerun()

def forgot_password_page():
    st.subheader("Forgot Password")
    st.caption("Both methods accept username or email.")

    tab1, tab2 = st.tabs(["Security Question", "Email OTP"])

    with tab1:
        with st.form("security_reset"):
            identity = st.text_input("Username or Email")
            question = st.selectbox("Security Question", [""] + SECURITY_QUESTIONS)
            answer = st.text_input("Security Answer")
            new_password = st.text_input("New Password", type="password")
            confirm = st.text_input("Confirm Password", type="password")
            submitted = st.form_submit_button("Reset Password", use_container_width=True)

        if submitted:
            user = get_user_by_identity(identity)
            if not all([identity, question, answer, new_password, confirm]):
                st.error("All fields are mandatory.")
            elif not user:
                st.error("User not found.")
            elif user[3] != question or not check_hash(answer.lower().strip(), user[4]):
                st.error("Security verification failed.")
            elif password_error(new_password):
                st.error(password_error(new_password))
            elif new_password != confirm:
                st.error("Passwords do not match.")
            elif password_was_used(user[0], new_password):
                st.error("New password cannot reuse an old password.")
            else:
                update_password(user[0], new_password)
                st.success("Password reset successful. Please login.")

    with tab2:
        identity = st.text_input("Username or Email", key="otp_identity")
        if st.button("Send OTP", use_container_width=True):
            user = get_user_by_identity(identity)
            if not identity:
                st.error("Enter username or email.")
            elif not user:
                st.error("Registered user not found.")
            else:
                email = user[0]
                otp = str(secrets.randbelow(900000) + 100000)
                st.session_state.otp = otp
                st.session_state.otp_email_value = email
                st.session_state.otp_expiry = time.time() + OTP_EXPIRY_SECONDS
                try:
                    send_otp_email(email, otp)
                    st.success(f"OTP sent to registered email: {email}")
                except Exception as e:
                    st.error("OTP email was not sent. Check EMAIL_ADDRESS and EMAIL_PASSWORD in Colab Secrets.")

        with st.form("otp_reset"):
            otp_value = st.text_input("OTP Verification")
            new_password = st.text_input("New Password", type="password", key="otp_new")
            confirm = st.text_input("Confirm Password", type="password", key="otp_confirm")
            submitted = st.form_submit_button("Verify OTP and Reset", use_container_width=True)

        if submitted:
            email_for_reset = st.session_state.get("otp_email_value", "")
            if time.time() > st.session_state.get("otp_expiry", 0):
                st.error("OTP expired.")
            elif otp_value != st.session_state.get("otp", ""):
                st.error("OTP verification failed.")
            elif password_error(new_password):
                st.error(password_error(new_password))
            elif new_password != confirm:
                st.error("Passwords do not match.")
            elif password_was_used(email_for_reset, new_password):
                st.error("New password cannot reuse an old password.")
            else:
                update_password(email_for_reset, new_password)
                st.success("Password reset successful. Please login.")

    if st.button("Back to Login"):
        st.session_state.page = "Login"; st.rerun()

def user_dashboard():
    token_data = verify_token(st.session_state.get("jwt_token", ""))
    if not token_data:
        st.error("Session expired. Please login again.")
        st.session_state.page = "Login"
        st.stop()

    user = get_user_by_email(token_data["email"])
    st.markdown(f"""
    <div class="card">
      <div class="eyebrow">User Dashboard</div>
      <h1>Welcome, {user[1]}</h1>
      <p>{user[0]}</p>
    </div>
    <div class="dash-grid">
      <div class="dash-card"><h3>JWT Session Status</h3><span class="status-pill">Active and valid</span></div>
      <div class="dash-card"><h3>Recent Login</h3><p>{user[6]}</p></div>
      <div class="dash-card"><h3>Notification</h3><p>Quote workspace is secured and ready.</p></div>
    </div>
    """, unsafe_allow_html=True)

    if st.button("Logout", use_container_width=True):
        st.session_state.pop("jwt_token", None)
        st.session_state.pop("user", None)
        st.session_state.page = "Login"
        st.rerun()

def admin_login():
    st.subheader("Admin Login")
    with st.form("admin_login_form"):
        username = st.text_input("Admin Username")
        password = st.text_input("Admin Password", type="password")
        submitted = st.form_submit_button("Enter Admin Console", use_container_width=True)

    if submitted:
        if username.strip().lower() == ADMIN_USERNAME.lower() and password.strip() == ADMIN_PASSWORD:
            st.session_state.admin = True
            st.success("Admin console unlocked.")
            time.sleep(1)
            st.session_state.page = "Admin Dashboard"
            st.rerun()
        else:
            st.error("Invalid admin credentials.")

    if st.button("Home"):
        st.session_state.page = "Landing Page"; st.rerun()

def admin_dashboard():
    if not st.session_state.get("admin"):
        st.error("Admin login required.")
        st.session_state.page = "Admin Login"
        st.stop()

    st.subheader("Admin Dashboard")
    st.caption("Passwords and password hashes are never displayed.")
    users = get_all_users()
    st.dataframe(
        [{"Username": u[0], "Email": u[1], "Created At": u[2], "Recent Login": u[3]} for u in users],
        use_container_width=True,
        hide_index=True
    )

    if st.button("Admin Logout"):
        st.session_state.admin = False
        st.session_state.page = "Admin Login"
        st.rerun()

init_db()
load_css()

if "page" not in st.session_state:
    st.session_state.page = "Landing Page"

page = st.session_state.page
if page == "Landing Page":
    landing_page()
elif page == "Login":
    login_page()
elif page == "Signup":
    signup_page()
elif page == "Forgot Password":
    forgot_password_page()
elif page == "User Dashboard":
    user_dashboard()
elif page == "Admin Login":
    admin_login()
elif page == "Admin Dashboard":
    admin_dashboard()

Writing app.py


In [3]:
from pyngrok import ngrok
from google.colab import userdata
import os, time

!pkill -f streamlit
!pkill -f ngrok
ngrok.kill()

ngrok_token = userdata.get("NGROK_AUTHTOKEN")
if not ngrok_token:
    raise Exception("Add NGROK_AUTHTOKEN in Colab Secrets first.")

ngrok.set_auth_token(ngrok_token)

os.environ["JWT_SECRET"] = userdata.get("JWT_SECRET") or "demo-secret"
os.environ["EMAIL_ADDRESS"] = userdata.get("EMAIL_ADDRESS") or ""
os.environ["EMAIL_PASSWORD"] = userdata.get("EMAIL_PASSWORD") or ""

get_ipython().system_raw(
    "streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &"
)

time.sleep(5)

public_url = ngrok.connect(8501, "http")
print("Open your FreightIQ app here:", public_url.public_url)

Open your FreightIQ app here: https://retold-distort-magnolia.ngrok-free.dev


In [5]:
from pyngrok import ngrok
from google.colab import userdata
import os, time

!pkill -f streamlit
!pkill -f ngrok
ngrok.kill()

ngrok_token = userdata.get("NGROK_AUTHTOKEN")
if not ngrok_token:
    raise Exception("Add NGROK_AUTHTOKEN in Colab Secrets first.")

ngrok.set_auth_token(ngrok_token)

os.environ["JWT_SECRET"] = userdata.get("JWT_SECRET") or "demo-secret"
os.environ["EMAIL_ADDRESS"] = userdata.get("EMAIL_ADDRESS") or ""
os.environ["EMAIL_PASSWORD"] = userdata.get("EMAIL_PASSWORD") or ""

print("EMAIL_ADDRESS loaded:", bool(os.environ["EMAIL_ADDRESS"]))
print("EMAIL_PASSWORD loaded:", bool(os.environ["EMAIL_PASSWORD"]))

get_ipython().system_raw(
    "streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &"
)

time.sleep(5)

public_url = ngrok.connect(8501, "http")
print("Open your FreightIQ app here:", public_url.public_url)

EMAIL_ADDRESS loaded: True
EMAIL_PASSWORD loaded: True
Open your FreightIQ app here: https://retold-distort-magnolia.ngrok-free.dev
